In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import matplotlib.pyplot as plt

def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
town = "Bonn"
objective = "cases_and_conc_pred_21d_conc"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results"

In [ ]:
phase_cut_date = "2023-03-15"
description = "50 weeks"

#phase_cut_dates = ["2024-01-03", "2023-07-19", "2023-03-15"]
#descriptions = ["8 weeks", "32 weeks", "50 weeks"]

In [ ]:
# load pred
conc_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_log_concentration.npz")
I_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_I7_reported.npz")

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": "Bonn",
            "sampling_area": "North_South",
            "project": "both", # one of ESI_CorA, AMELAG
            "max_precipitation_subsetting": None, # one of None, dry, light_rain
            "substance_normalization": "flow", # one of None, PMMoV, flow
            "gene_target": "N1", # one of N1, N2
            "log_scale": True, # this only considers WW measurements, not case counts
        },

        "E0": 862.857, 
        "I0": 1294.286,
        "R0": 162092.04, # 92% of pop, based on https://www.rki.de/DE/Themen/Infektionskrankheiten/Infektionskrankheiten-A-Z/C/COVID-19-Pandemie/AK-Studien/Ergebnisse.html
        "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "underreporting_model": "monotone_increasing",
        "n_days_pred_conc": 3 * 7, # number of days to predict into the future
}



In [ ]:
import matplotlib.dates as mdates
import pandas as pd
import matplotlib.pyplot as plt


fig = plt.figure(figsize=(5.1, 4.2), dpi=300, constrained_layout=True)
gs = fig.add_gridspec(nrows=2, ncols=2, width_ratios=[4.1, 1.6])

ax_top  = fig.add_subplot(gs[0, 0])
ax_bot  = fig.add_subplot(gs[1, 0], sharex=ax_top)
ax_zoom = fig.add_subplot(gs[:, 1], sharey=ax_top)  # spans both rows, shares y with top

# (optional) reduce clutter: hide left ticks on zoom panel
ax_zoom.tick_params(axis="y", left=False, labelleft=False)

# -------------------------------------------------------
# your existing code, but replace axs[0] -> ax_top and axs[1] -> ax_bot
# -------------------------------------------------------

hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_{objective}/hparams.json"

with open(hparams_path) as f:
    hparams = json.load(f)

base_config.update(hparams)
base_config["phase_cut_date"] = phase_cut_date
data = optimization_utils.two_phase_integrative_model_load_data(base_config)

quantiles = {q: jnp.quantile(conc_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

conc_median = quantiles[0.5]
conc_low  = quantiles[0.025]
conc_high = quantiles[0.975]

# --- TOP (left) ---
ax_top.scatter(data["conc_dates_train"], data["conc_train"],
               label="Training/validation\n(concentration)", color="#3d85c6ff", alpha=0.7, s=15)
ax_top.scatter(data["conc_dates_val"], data["conc_val"],
               label=None, color="#3d85c6ff", alpha=0.7, s=15)
ax_top.scatter(data["conc_dates_test"], data["test_conc"],
               label="Test", color="#595959", alpha=0.7, s=15)

split_date = pd.to_datetime(data["conc_dates_train"][-1])
ax_top.axvline(split_date, color="#595959", linestyle="--", label="Phase split")

valid_idx = (data["t_all_idx"] * base_config["dt"] >= int(hparams.get("T_max")))
ax_top.fill_between(data["dates_all"][valid_idx], conc_low, conc_high,
                    color="#8B0000", alpha=0.15, label="95% CI")
conc_low  = quantiles[0.05]
conc_high = quantiles[0.95]
ax_top.fill_between(data["dates_all"][valid_idx], conc_low, conc_high,
                    color="#8B0000", alpha=0.3, label="90% CI")
conc_low  = quantiles[0.25]
conc_high = quantiles[0.75]
ax_top.fill_between(data["dates_all"][valid_idx], conc_low, conc_high,
                    color="#8B0000", alpha=0.45, label="50% CI")
ax_top.plot(data["dates_all"][valid_idx], conc_median, label="Median", color="#8B1000")


# --- BOTTOM (left) ---
quantiles_I = {q: jnp.quantile(I_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
I7_median = quantiles_I[0.5]
I7_low  = quantiles_I[0.025]
I7_high = quantiles_I[0.975]

ax_bot.scatter(data["I_dates_train"], data["I_train"],
               label="Training/validation\n(reported cases)", color="goldenrod", alpha=0.7, s=15)
ax_bot.scatter(data["I_dates_val"], data["I_val"],
               label=None, color="goldenrod", alpha=0.7, s=15)
ax_bot.scatter(data["obs_dates_phase_2"], data["I_test"],
               label="Test", color="#595959", alpha=0.7, s=15)
ax_bot.axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle="--", label="Phase split")

ax_bot.fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.15, label="95% CI")
I7_low  = quantiles_I[0.05]
I7_high = quantiles_I[0.95]
ax_bot.fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.3, label="90% CI")
I7_low  = quantiles_I[0.25]
I7_high = quantiles_I[0.75]
ax_bot.fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.45, label="50% CI")
ax_bot.plot(data["dates_all"][1:], I7_median, label="Median", color="#8B1000")


# --- NEW: ZOOM PANEL (repeat TOP plot, then set x-limits) ---
# Re-plot the same elements as the top axis:
ax_zoom.scatter(data["conc_dates_train"], data["conc_train"], color="#3d85c6ff", alpha=0.7, s=15)
ax_zoom.scatter(data["conc_dates_val"], data["conc_val"], color="#3d85c6ff", alpha=0.7, s=15)
ax_zoom.scatter(data["conc_dates_test"], data["test_conc"], color="#595959", alpha=0.7, s=15)


ax_zoom.axvline(split_date, color="#595959", linestyle="--")

# Use the same quantile bands/median as on ax_top
conc_low  = quantiles[0.025]
conc_high = quantiles[0.975]
ax_zoom.fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.15)

conc_low  = quantiles[0.05]
conc_high = quantiles[0.95]
ax_zoom.fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.3)

conc_low  = quantiles[0.25]
conc_high = quantiles[0.75]
ax_zoom.fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.45)

ax_zoom.plot(data["dates_all"][valid_idx], conc_median, color="#8B1000")

# Choose zoom window:
# - if you want *strictly after* the dashed line, set zoom_start = split_date
# - if you want a little context, subtract e.g. 30 days
zoom_start = split_date - pd.Timedelta(days=30)
zoom_end = pd.to_datetime(data["dates_all"][-1]) + pd.Timedelta(days=3)
ax_zoom.set_xlim(zoom_start, zoom_end)

# Formatting
ax_top.set_ylabel("Flow norm.\nconcentration\n[log(GU/(ld))]")
ax_bot.set_ylabel("7-day moving\nsum of new\ninfections [#]")

# Left column x-axis formatting
ax_top.tick_params(axis="x", labelbottom=False)

ax_bot.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
ax_bot.tick_params(axis='x', rotation=45)

# Zoom axis formatting (keep it simple)
ax_zoom.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
ax_zoom.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
ax_zoom.tick_params(axis='x', rotation=45)

# No plt.tight_layout() needed with constrained_layout=True
plt.tight_layout()
plt.savefig("Bonn_pred.png", dpi=300, bbox_inches="tight")


In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 5), dpi=300)

handles, labels = [], []
for ax in axs.ravel():
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll and ll not in labels:
            labels.append(ll)
            handles.append(hh)
order = [0, 6, 7, 1, 5, 4, 3, 2]
labels = [labels[i] for i in order]
handles = [handles[i] for i in order]

ax2.axis('off')
fig2.legend(handles, labels, frameon=False)

fig2.savefig("Bonn_pred_legend.png", dpi=300)